# QML Air Quality — ablação angular (Q01–Q10)

**Settings Kaggle:** Internet ON · Accelerator None (CPU) · sessão longa.

**Dataset:** anexe o dataset do código (o Kaggle costuma **descompactar** o zip sozinho — isso é normal).  
Opcional: dataset do cache Q01–Q03.

Na 1ª célula, confira o print `Datasets em /kaggle/input:` — deve listar o nome do seu dataset.

In [ ]:
import os, zipfile, shutil, sys
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
ROOT = WORK / 'qml_air_quality_poc'
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

print('Datasets em /kaggle/input:')
for p in sorted(INPUT.iterdir()):
    print(' -', p.name)


def copy_tree(src: Path, dst: Path) -> None:
    dst.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        target = dst / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)


# --- código: zip OU já extraído pelo Kaggle ---
zips = list(INPUT.rglob('qml_air_quality_poc.zip'))
src_markers = list(INPUT.rglob('qml_air_quality/config.py'))  # .../src/qml_air_quality/config.py
cfg_markers = list(INPUT.rglob('quantum_ablation.yaml'))

if zips:
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(ROOT)
    print('Extraído do zip:', zips[0])
elif src_markers:
    # Kaggle descompactou o zip: achar a pasta que contém src/ e config/
    pkg = src_markers[0].parents[1]  # .../src
    code_root = pkg.parent            # pasta com src/ e (idealmente) config/
    print('Dataset já extraído em:', code_root)
    copy_tree(code_root, ROOT)
elif cfg_markers:
    code_root = cfg_markers[0].parents[1]  # pasta acima de config/
    print('Dataset via config em:', code_root)
    copy_tree(code_root, ROOT)
else:
    raise FileNotFoundError(
        'Não achei o código. Em /kaggle/input deve existir '
        'qml_air_quality_poc.zip OU pastas src/ + config/ já extraídas. '
        f'Conteúdo visto: {[p.name for p in INPUT.iterdir()]}'
    )

# se o zip tinha prefixo extra, achar src de verdade
if not (ROOT / 'src' / 'qml_air_quality').exists():
    hits = list(ROOT.rglob('src/qml_air_quality'))
    if hits:
        real = hits[0].parents[1]
        print('Ajustando ROOT aninhado:', real)
        tmp = WORK / '_poc_tmp'
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(real, tmp)
        shutil.rmtree(ROOT)
        tmp.rename(ROOT)

assert (ROOT / 'src' / 'qml_air_quality').exists(), f'src ausente em {list(ROOT.iterdir())}'
assert (ROOT / 'config' / 'quantum_ablation.yaml').exists(), f'config ausente em {list(ROOT.iterdir())}'
print('OK código em', ROOT)
print('top:', sorted(p.name for p in ROOT.iterdir()))

# --- cache de kernels (opcional) ---
ker_dir = ROOT / 'artifacts' / 'ablation' / 'kernels'
ker_dir.mkdir(parents=True, exist_ok=True)

cache_zips = list(INPUT.rglob('ablation_kernel_cache*.zip'))
npy_hits = list(INPUT.rglob('Q01_seed42_K_train.npy'))

if cache_zips:
    with zipfile.ZipFile(cache_zips[0]) as zf:
        zf.extractall(WORK / '_cache_extract')
    search_root = WORK / '_cache_extract'
elif npy_hits:
    search_root = npy_hits[0].parent
    print('Cache já extraído em', search_root)
else:
    search_root = None

if search_root is not None:
    for p in search_root.rglob('Q0*_seed42_*'):
        if p.suffix in {'.npy', '.json'}:
            shutil.copy2(p, ker_dir / p.name)
    print('Cached kernels:', sorted(x.name for x in ker_dir.glob('Q0*'))[:15])
else:
    print('Sem cache de kernels — calcula Q01–Q10 do zero (~2–3 h).')

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('CWD', Path.cwd())
print('sys.path[0]', sys.path[0])

In [ ]:
%pip install -q qiskit qiskit-machine-learning qiskit-aer scikit-learn pandas numpy matplotlib pyyaml joblib

import qiskit, qiskit_machine_learning, sklearn
print('qiskit', qiskit.__version__)
print('qiskit-ml', qiskit_machine_learning.__version__)
print('sklearn', sklearn.__version__)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

from qml_air_quality.experiments.quantum_ablation import run_ablation
from qml_air_quality.reporting.ablation_report import plot_ablation_figures, write_ablation_report

# force_kernels=False reutiliza Q01–Q03 se o cache estiver em artifacts/ablation/kernels/
result = run_ablation(force_kernels=False, skip_test=False)

display(result['validation'])
print('selection:', result['selection'])
if 'summary' in result:
    print('classification:', result['summary']['classification'])
    display(result['test'])

In [ ]:
report = write_ablation_report()
plots = plot_ablation_figures()
print('Report:', report)
for p in plots:
    print(p)

# compactar saída para download
import zipfile
out_zip = Path('/kaggle/working/ablation_results.zip')
abl = Path('artifacts/ablation')
with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in abl.rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to(abl.parent.parent))
print('Wrote', out_zip, 'size_mb', out_zip.stat().st_size / 1e6)